# A1 Model checks

Four checks on the prediction models: whether the learner matters, whether weighting every user equally changes the results, and whether creator features or the variety of clicked topics add anything. Outputs: `11_compare_*`, `12_*`.

The slow steps are functions in `src/thesis/heavy/checks.py`, read through `ensure(step)`.

## 1. Setup

In [1]:
# Libraries and shared project code
import pandas as pd
from IPython.display import display
from thesis.cache import ensure

## 2. Learners

Logistic regression, LightGBM, random forest and XGBoost on the same features and splits, with settings chosen on validation users.

In [2]:
# Logistic regression, LightGBM, random forest and XGBoost for both outcomes: validation scores (model choice), then test
# scores and differences from LightGBM (thesis.heavy.checks.compare)
C11 = ensure("A1.compare")
RES = {t: C11[f"11_compare_{t}"] for t in ("R", "C")}
display(pd.concat({t: pd.DataFrame(RES[t]["validation"]).T for t in RES}))
pd.concat({t: pd.DataFrame(RES[t]["models"]).T for t in RES})

[A1.compare] saved results loaded


pseudo_r2     auc
R logistic          0.2043  0.7804
  lightgbm          0.2197  0.7921
  random_forest     0.2177  0.7908
  xgboost           0.2196  0.7920
C logistic          0.0756  0.7507
  lightgbm          0.0786  0.7685
  random_forest     0.0783  0.7703
  xgboost           0.0785  0.7680

pseudo_r2     auc r2_minus_lightgbm_ci95  \
R logistic         0.2524  0.8169     [-0.0088, -0.0072]   
  lightgbm         0.2605  0.8191             [0.0, 0.0]   
  random_forest    0.2592  0.8193     [-0.0017, -0.0008]   
  xgboost          0.2602  0.8189     [-0.0006, -0.0001]   
C logistic         0.0896  0.8035     [-0.0026, -0.0011]   
  lightgbm         0.0916  0.8121             [0.0, 0.0]   
  random_forest    0.0915  0.8111      [-0.0005, 0.0003]   
  xgboost          0.0915  0.8126      [-0.0003, 0.0001]   

                auc_minus_lightgbm_ci95  
R logistic           [-0.0028, -0.0017]  
  lightgbm                   [0.0, 0.0]  
  random_forest       [-0.0001, 0.0004]  
  xgboost            [-0.0004, -0.0001]  
C logistic           [-0.0104, -0.0068]  
  lightgbm                   [0.0, 0.0]  
  random_forest       [-0.002, -0.0003]  
  xgboost              [-0.0001, 0.001]

In [3]:
# Fit time in seconds (each includes the learner's own setting search: XGBoost and the random forest fit several)
pd.DataFrame({t: RES[t]["seconds"] for t in RES})

,R,C
logistic,11.0,6.6
lightgbm,52.1,15.6
random_forest,276.0,162.6
xgboost,630.6,424.1


In [4]:
# Training curves: test AUC against training time for each learner, refitted once with the settings chosen above.
# Time counts training only; the AUC at each checkpoint is computed afterwards (thesis.heavy.checks.training_curve)
CURVE = ensure("A1.training_curve")["11_training_curve"]
CURVE.groupby(["target", "model"]).agg(points=("trees", "size"), seconds=("seconds", "max"), final_auc=("test_auc", "last"))

[A1.training_curve] saved results loaded


points  seconds  final_auc
target model                                    
C      lightgbm           33    13.03    0.81215
       logistic            1     6.28    0.80355
       random_forest      20   113.00    0.81106
       xgboost            33    20.78    0.81256
R      lightgbm           34    45.66    0.81915
       logistic            1    10.85    0.81687
       random_forest      20   113.46    0.81929
       xgboost            33    45.23    0.81889

## 3. User weights in training
LightGBM for return with and without equal user weights.

In [5]:
# LightGBM with and without equal user weights (thesis.heavy.checks.weighting)
res = ensure("A1.weighting")["12_auc_experiments"]
pd.DataFrame({"auc": res["auc"], "minus_no_user_weight_ci95": res["minus_no_user_weight_ci95"]})

[A1.weighting] saved results loaded


,auc,minus_no_user_weight_ci95
no_user_weight,0.8191,"[0.0, 0.0]"
equal_user_weight,0.8186,"[-0.0007, -0.0005]"


## 4. Creator features
Do the creators behind the cards shown, or the user's ties to them, add anything to the full LightGBM models?

In [6]:
# Creator features per snapshot, from information up to the origin day (creator statistics up to the day before);
# full LightGBM with and without them, overall and by days since first seen (thesis.heavy.checks.creator)
CRES = ensure("A1.creator")["12_creator_features"]
display(pd.DataFrame(CRES["describe"]).T if "describe" in CRES else pd.Series(CRES["mean_all_user_days"], name="mean"))
pd.concat({t: pd.DataFrame({s: v for s, v in CRES[t].items() if s != "creator_share_of_gain"}).T for t in ("R", "C")})

[A1.creator] saved results loaded


,count,mean,std,min,25%,50%,75%,max
h_cr_from_clicked_share,4306153.0,0.0037,0.0151,0.0000,0.0000,0.0000,0.0000,0.4348
w7_cr_from_clicked_share,4306153.0,0.0053,0.0240,0.0000,0.0000,0.0000,0.0000,1.0000
h_cr_ctr_shown,4306153.0,0.0500,0.0090,0.0025,0.0442,0.0494,0.0546,0.1938
h_cr_posts_shown,4306153.0,1.0628,0.6631,0.0000,0.5985,0.9986,1.4233,6.3509
h_cr_distinct_clicked,4306153.0,2.2427,11.8796,0.0000,0.0000,0.0000,1.0000,1639.0000


rows pseudo_r2_all_59 auc_all_59 pseudo_r2_plus_creator  \
R all           228295           0.2605     0.8191                  0.263   
  history_1      63602           0.2616     0.5538                 0.2685   
  history_2-7    23384           0.0641     0.6524                 0.0699   
  history_8-14   39223           0.1158     0.7253                 0.1153   
  history_15+   102086            0.371     0.8431                 0.3703   
C all           129201           0.0916     0.8121                 0.0912   
  history_1      16189           0.0527     0.6373                 0.0512   
  history_2-7    11869           0.0737     0.7246                 0.0729   
  history_8-14   22437           0.0827     0.7738                  0.082   
  history_15+    78706            0.096     0.8262                 0.0958   

               auc_plus_creator pseudo_r2_gain_ci95      auc_gain_ci95  
R all                    0.8204    [0.0022, 0.0028]    [0.001, 0.0014]  
  history_1              0.5557    [0.0063, 0.0076]   [0.0003, 0.0035]  
  history_2-7             0.653    [0.0047, 0.0068]  [-0.0009, 0.0021]  
  history_8-14           0.7256   [-0.0011, 0.0001]  [-0.0002, 0.0008]  
  history_15+             0.843   [-0.001, -0.0002]  [-0.0004, 0.0001]  
C all                    0.8127  [-0.0006, -0.0001]    [0.0001, 0.001]  
  history_1              0.6257  [-0.0023, -0.0003]  [-0.0175, -0.005]  
  history_2-7            0.7245      [-0.0016, 0.0]  [-0.0028, 0.0025]  
  history_8-14           0.7741  [-0.0014, -0.0001]  [-0.0012, 0.0015]  
  history_15+            0.8268      [-0.0004, 0.0]   [0.0001, 0.0011]

## 5. Variety of what the user clicks
Does the variety of topics a user has clicked add anything to the full LightGBM models?

In [7]:
# Variety of what the user clicked (distinct topics over the whole history and the last 7 days, topics per click);
# full LightGBM with and without it (thesis.heavy.checks.variety)
VRES = ensure("A1.variety")["12_variety_features"]
display(pd.DataFrame(VRES["describe"]).T if "describe" in VRES else pd.Series(VRES["mean_all_user_days"], name="mean"))
pd.concat({t: pd.DataFrame({s: v for s, v in VRES[t].items() if s != "extra_share_of_gain"}).T for t in ("R", "C")})

[A1.variety] saved results loaded


,count,mean,std,min,25%,50%,75%,max
h_topics_clicked,4306153.0,1.7724,7.4916,0.0,0.0,0.0,1.0000,426.0
w7_topics_clicked,4306153.0,1.1321,4.8470,0.0,0.0,0.0,0.0000,352.0
h_topics_per_click,4306153.0,0.2506,0.4196,0.0,0.0,0.0,0.6667,1.0


rows pseudo_r2_all_59 auc_all_59 pseudo_r2_plus_extra  \
R all           228295           0.2605     0.8191               0.2609   
  history_1      63602           0.2616     0.5538               0.2618   
  history_2-7    23384           0.0641     0.6524                0.064   
  history_8-14   39223           0.1158     0.7253               0.1165   
  history_15+   102086            0.371     0.8431               0.3716   
C all           129201           0.0916     0.8121               0.0915   
  history_1      16189           0.0527     0.6373               0.0522   
  history_2-7    11869           0.0737     0.7246                0.074   
  history_8-14   22437           0.0827     0.7738               0.0825   
  history_15+    78706            0.096     0.8262                0.096   

               auc_plus_extra pseudo_r2_gain_ci95      auc_gain_ci95  
R all                  0.8192    [0.0003, 0.0006]  [-0.0001, 0.0001]  
  history_1            0.5542       [0.0, 0.0004]  [-0.0002, 0.0011]  
  history_2-7          0.6522   [-0.0006, 0.0005]  [-0.0011, 0.0008]  
  history_8-14         0.7258    [0.0003, 0.0012]   [0.0001, 0.0009]  
  history_15+          0.8433    [0.0003, 0.0009]     [-0.0, 0.0003]  
C all                  0.8127   [-0.0002, 0.0002]   [0.0002, 0.0008]  
  history_1            0.6372   [-0.0012, 0.0001]   [-0.003, 0.0029]  
  history_2-7          0.7254   [-0.0003, 0.0009]   [-0.001, 0.0025]  
  history_8-14         0.7739   [-0.0006, 0.0003]   [-0.0009, 0.001]  
  history_15+          0.8268   [-0.0002, 0.0003]   [0.0002, 0.0009]

## 6. Summary

The tree ensembles match LightGBM and the logistic regression is slightly below. Creator and variety features add little.